In [ ]:
%%file producer.py
from kafka import KafkaProducer
import json, random, time
from datetime import datetime

producer = KafkaProducer(
    bootstrap_servers='broker:9092',
    value_serializer=lambda v: json.dumps(v).encode('utf-8')
)

sklepy = ['Warszawa', 'Kraków', 'Gdańsk', 'Wrocław']
kategorie = ['elektronika', 'odzież', 'żywność', 'książki']

def generate_transaction():
    return {
        'tx_id': f'TX{random.randint(1000,9999)}',
        'user_id': f'u{random.randint(1,20):02d}',
        'amount': round(random.uniform(5.0, 5000.0), 2),
        'store': random.choice(sklepy),
        'category': random.choice(kategorie),
        'timestamp': datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
    }

for i in range(1000):
    tx = generate_transaction()
    producer.send('transactions', value=tx)
    print(f"[{i+1}] {tx['tx_id']} | {tx['amount']:.2f} PLN | {tx['store']}")
    time.sleep(0.5)

producer.flush()
producer.close()

In [ ]:
import os
import pyspark

spark_version = pyspark.__version__
print(f"Detected PySpark version: {spark_version}")

if spark_version.startswith("4"):
    KAFKA_PACKAGE = "org.apache.spark:spark-sql-kafka-0-10_2.13:4.0.0-preview2"
else:
    KAFKA_PACKAGE = "org.apache.spark:spark-sql-kafka-0-10_2.12:3.5.0"

print(f"Using connector: {KAFKA_PACKAGE}")

#
os.environ['PYSPARK_SUBMIT_ARGS'] = f'--packages {KAFKA_PACKAGE} pyspark-shell'

In [ ]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("Lab4-Kafka")
    .config("spark.jars.packages", KAFKA_PACKAGE)
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")
print(f"Spark {spark.version} — ready")

In [ ]:
from pyspark.sql.functions import col

kafka_raw = (
    spark.readStream
    .format("kafka")
    .option("kafka.bootstrap.servers", "broker:9092")
    .option("subscribe", "transactions")
    .option("startingOffsets", "earliest")
    .load()
)

kafka_raw.printSchema()

In [ ]:
%%writefile kafka_raw.py
from pyspark.sql import SparkSession
from pyspark.sql.functions import col

spark = (
    SparkSession.builder
    .appName("Lab4-Kafka")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")

kafka_raw = (
    spark.readStream
    .format("kafka")
    .option("kafka.bootstrap.servers", "broker:9092")
    .option("subscribe", "transactions")
    .option("startingOffsets", "earliest")
    .load()
)

df = kafka_raw.select(col("value").cast("string"))
q = (
    df.writeStream
    .format("console")
    .outputMode("append")
    .option("truncate", False)
    .start()
)

q.awaitTermination()

In [ ]:
%%writefile test_file.py
print("hello")

In [ ]:
from pyspark.sql.types import StructType, StructField, StringType, DoubleType
from pyspark.sql.functions import from_json, col, to_timestamp

tx_schema = StructType([
    StructField("tx_id", StringType()),
    StructField("user_id", StringType()),
    StructField("amount", DoubleType()),
    StructField("store", StringType()),
    StructField("category", StringType()),
    StructField("timestamp", StringType()),
])

df = (
    kafka_raw
    .select(from_json(col("value").cast("string"), tx_schema).alias("tx"))
    .select("tx.*")
    .withColumn("timestamp", to_timestamp("timestamp", "yyyy-MM-dd HH:mm:ss"))
)

df.printSchema()

In [ ]:
batch_counter = {"n": 0}

def show_parsed(batch_df, batch_id):
    batch_counter["n"] += 1
    print(f"--- Batch {batch_id} ---")
    batch_df.show(5, truncate=False)

    if batch_counter["n"] >= 2:
        raise Exception("stop")

q = (
    df.writeStream
    .foreachBatch(show_parsed)
    .option("checkpointLocation", "/tmp/chk_lab4_parsed")
    .start()
)

try:
    q.awaitTermination()
except:
    q.stop()

In [ ]:
#Task 4: tumbling window per store
from pyspark.sql.functions import window, count, sum as _sum, round as _round

windowed = (
    df
    .withWatermark("timestamp", "30 seconds")
    .groupBy(window("timestamp", "1 minute"), "store")
    .agg(
        count("tx_id").alias("tx_count"),
        _round(_sum("amount"), 2).alias("total_amount")
    )
)

batch_counter = {"n": 0}

def show_window(batch_df, batch_id):
    batch_counter["n"] += 1
    print(f"\n=== Batch {batch_id} ===")
    (
        batch_df
        .select(
            col("window.start").alias("from"),
            col("window.end").alias("to"),
            "store",
            "tx_count",
            "total_amount"
        )
        .orderBy("from", "store")
        .show(truncate=False)
    )

    if batch_counter["n"] >= 5:
        raise Exception("stop")

q = (
    windowed.writeStream
    .outputMode("append")
    .foreachBatch(show_window)
    .option("checkpointLocation", "/tmp/chk_lab4_windows_store")
    .start()
)

try:
    q.awaitTermination()
except:
    q.stop()

In [ ]:
#Task 4.2 — window per category, complete mode
windowed_cat = (
    df
    .withWatermark("timestamp", "30 seconds")
    .groupBy(window("timestamp", "1 minute"), "category")
    .agg(
        count("tx_id").alias("tx_count"),
        _round(_sum("amount"), 2).alias("total_amount")
    )
)

batch_counter = {"n": 0}

def show_window_category(batch_df, batch_id):
    batch_counter["n"] += 1
    print(f"\n=== Batch {batch_id} ===")
    (
        batch_df
        .select(
            col("window.start").alias("from"),
            col("window.end").alias("to"),
            "category",
            "tx_count",
            "total_amount"
        )
        .orderBy("from", "category")
        .show(truncate=False)
    )

    if batch_counter["n"] >= 5:
        raise Exception("stop")

q = (
    windowed_cat.writeStream
    .outputMode("complete")
    .foreachBatch(show_window_category)
    .option("checkpointLocation", "/tmp/chk_lab4_windows_category")
    .start()
)

try:
    q.awaitTermination()
except:
    q.stop()

In [ ]:
alert_query.stop()

In [ ]:
#Homework: What happens when you stop the producer and wait 2 minutes?
#When the producer stops, Kafka receives no new transactions. Spark may still process messages already stored in Kafka, but after that no new records arrive.

#For windowed aggregations in append mode, Spark only outputs a window when the watermark says the window is finished. If no new events arrive, the watermark may stop moving, so some final windows may not be emitted immediately. Therefore, after stopping the producer and waiting 2 minutes, the output becomes quiet, and some window results may remain delayed until new data arrives or the query is stopped.

# So no producer means no new data; without new event times, the watermark does not progress normally, so append-mode window results can be delayed.

In [ ]:
##Task 5.2 Review Questions
###1. Why do the first results in append mode appear with a delay?
#Because in append mode, Spark only outputs a window after it is sure the window is complete. With event-time windows and a watermark, Spark waits for late-arriving data before closing the window. Therefore, the first results do not appear immediately.

###2. What happens if you set the watermark to "0 seconds"?
#Spark will not wait for late data. It closes windows as soon as possible based on event time. This can make results appear faster, but late records may be ignored or dropped from already-closed windows.

###3. What is the difference between window() in Lab 2 batch and here in streaming?
#The window logic is similar, but in batch all data is already available, so Spark can compute all windows at once. In streaming, data arrives continuously, so Spark updates windows over time and uses watermarks to decide when a window is complete.

###4. Why do we use to_json() when writing to Kafka instead of just select("value")?
#Kafka expects the output column value to be a string or bytes. After processing, our data is in Spark columns or a struct, not a Kafka-ready JSON message. to_json(struct(...)) converts the selected columns back into one JSON string that Kafka can store as the message value.

In [ ]:
#Homework: sliding window 2 minutes / 1-minute step
sliding_window = (
    df
    .withWatermark("timestamp", "30 seconds")
    .groupBy(
        window("timestamp", "2 minutes", "1 minute"),
        "store"
    )
    .agg(
        count("tx_id").alias("tx_count"),
        _round(_sum("amount"), 2).alias("total_amount")
    )
)

batch_counter = {"n": 0}

def show_sliding_window(batch_df, batch_id):
    batch_counter["n"] += 1
    print(f"\n=== Batch {batch_id} ===")
    (
        batch_df
        .select(
            col("window.start").alias("from"),
            col("window.end").alias("to"),
            "store",
            "tx_count",
            "total_amount"
        )
        .orderBy("from", "store")
        .show(truncate=False)
    )

    if batch_counter["n"] >= 5:
        raise Exception("stop")

q = (
    sliding_window.writeStream
    .outputMode("append")
    .foreachBatch(show_sliding_window)
    .option("checkpointLocation", "/tmp/chk_lab4_sliding_window")
    .start()
)

try:
    q.awaitTermination()
except:
    q.stop()

In [ ]:
alert_ratio_query.stop()

In [ ]:
from pyspark.sql.functions import to_json, struct, lit, round as _round, col

alerts_ratio = (
    df
    .filter(col("amount") > 3000)
    .select(
        to_json(
            struct(
                "tx_id",
                "user_id",
                "amount",
                _round((col("amount") / 400.0), 2).alias("ratio"),
                "store",
                "category",
                col("timestamp").cast("string").alias("timestamp"),
                lit("HIGH").alias("alert_level")
            )
        ).alias("value")
    )
)

alert_ratio_query = (
    alerts_ratio.writeStream
    .format("kafka")
    .option("kafka.bootstrap.servers", "broker:9092")
    .option("topic", "alerts")
    .option("checkpointLocation", "/tmp/chk_lab4_alerts_ratio")
    .outputMode("append")
    .start()
)

print("Alert stream with ratio started.")